In [2]:
import os
import time
import math
import geopandas as gpd
import pandas as pd
from rasterstats import zonal_stats

# -----------------------------
# USER INPUTS
# -----------------------------
shapefile_path = r"C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\CONUS\CONUS_3_3.shp"
ghi_raster = r"C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\CONUS\Non-Arctic_Average_GHI_y2005_2024_5070.tif"
log_file = r"C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\CONUS\GHI_CONUS.txt"
output_csv = r"C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\CONUS\CONUS_GHI.csv"
chunk_size = 10000

# -----------------------------
# Logging
# -----------------------------
def log(msg):
    print(msg)
    with open(log_file, "a") as f:
        f.write(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}\n")

# -----------------------------
# Remove old CSV if exists
# -----------------------------
if os.path.exists(output_csv):
    os.remove(output_csv)
    log("Existing output CSV removed.")

# -----------------------------
# Load shapefile
# -----------------------------
log(f"Loading shapefile: {shapefile_path}")
gdf = gpd.read_file(shapefile_path)

# Ensure required fields exist
required_fields = ["ROW_ID", "Square_Met"]
for field in required_fields:
    if field not in gdf.columns:
        raise ValueError(f"Missing required field: {field}")

total_polygons = len(gdf)
log(f"Loaded {total_polygons} polygons.")

# -----------------------------
# Compute chunks
# -----------------------------
num_chunks = math.ceil(total_polygons / chunk_size)
log(f"Processing in {num_chunks} chunks of {chunk_size} polygons each.")

# -----------------------------
# Process chunks
# -----------------------------
for chunk_idx in range(num_chunks):
    start_idx = chunk_idx * chunk_size
    end_idx = min((chunk_idx + 1) * chunk_size, total_polygons)
    chunk_gdf = gdf.iloc[start_idx:end_idx].copy()

    log(f"Processing chunk {chunk_idx + 1}/{num_chunks} ({start_idx} to {end_idx - 1})...")
    start_time = time.time()

    # Zonal statistics
    stats = zonal_stats(
        chunk_gdf,
        ghi_raster,
        stats=["mean"],
        all_touched=True,
        nodata=-9999
    )

    # Add GHI_mean
    chunk_gdf["GHI_mean"] = [s["mean"] for s in stats]

    # Keep only desired columns
    output_df = chunk_gdf[["ROW_ID", "Square_Met", "GHI_mean"]]

    # Append to CSV
    if chunk_idx == 0:
        output_df.to_csv(output_csv, index=False, mode="w")
    else:
        output_df.to_csv(output_csv, index=False, mode="a", header=False)

    elapsed = time.time() - start_time
    log(f"Chunk {chunk_idx + 1} processed in {elapsed:.2f} sec.")

log("All chunks processed successfully.")
log(f"Final CSV saved to: {output_csv}")

Loading shapefile: C:\Users\KyleSteen.AzureAD\Documents\GHI\GHI_Workspace\CONUS\CONUS_3_3.shp
Loaded 835996 polygons.
Processing in 84 chunks of 10000 polygons each.
Processing chunk 1/84 (0 to 9999)...
Chunk 1 processed in 44.03 sec.
Processing chunk 2/84 (10000 to 19999)...
Chunk 2 processed in 44.37 sec.
Processing chunk 3/84 (20000 to 29999)...
Chunk 3 processed in 44.95 sec.
Processing chunk 4/84 (30000 to 39999)...
Chunk 4 processed in 41.44 sec.
Processing chunk 5/84 (40000 to 49999)...
Chunk 5 processed in 43.17 sec.
Processing chunk 6/84 (50000 to 59999)...
Chunk 6 processed in 39.77 sec.
Processing chunk 7/84 (60000 to 69999)...
Chunk 7 processed in 41.72 sec.
Processing chunk 8/84 (70000 to 79999)...
Chunk 8 processed in 41.09 sec.
Processing chunk 9/84 (80000 to 89999)...
Chunk 9 processed in 42.38 sec.
Processing chunk 10/84 (90000 to 99999)...
Chunk 10 processed in 40.50 sec.
Processing chunk 11/84 (100000 to 109999)...
Chunk 11 processed in 40.96 sec.
Processing chunk 12